In [14]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.linear_model import LinearRegression, LogisticRegressionCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

In [17]:
track_history  = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/now_playing.csv")
track_analysis = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_analysis.csv")
track_metadata = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_metadata.csv")
weather        = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/weather.csv")

# convert all request times to datetime formats
for df in [track_history, track_analysis, track_metadata, weather]:
    df["request_time"] = pd.to_datetime(df["request_time"])

# converts a string in the format mm:ss to milliseconds
def mm_ss_to_ms(mm_ss_str):
    minutes, seconds = map(int, mm_ss_str.split(":"))
    return ((minutes * 60) + seconds) * 1000

# remove analysis/metadata observations whose id is null
track_analysis = track_analysis[~track_analysis["id"].isna()]
track_metadata = track_metadata[~track_metadata["id"].isna()]
# remove all metadata rows whose genres are null
track_metadata = track_metadata.dropna(subset=["genres"])
# if any values are missing in the track metadata (which is acceptable), set them to be an empty string
track_metadata.fillna("", inplace=True)
# change track analysis duration (mm:ss) to milliseconds
track_analysis["duration"] = track_analysis["duration"].apply(lambda d: mm_ss_to_ms(d))
# remove dB from track analysis loudness and convert to an int (e.g., "-2 dB" becomes -2)
track_analysis["loudness"] = track_analysis["loudness"].apply(lambda db: db.replace(" dB", "")).astype(np.int16)
# split camelot into three columns: X/y position via cos(camelot num * 2pi/12) and is_major
track_analysis["is_major"] = (track_analysis["camelot"].apply(lambda camelot: "B" in camelot)).astype(np.int16)
# turn camelot into coordinates (better for modeling; see explanation below)
track_analysis["camelot_x"] = np.cos(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)
track_analysis["camelot_y"] = np.sin(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)
# change is_ad to be a number, rather than a boolean calue
track_history["is_ad"] = track_history["is_ad"].astype(np.int16)

# keep only relevant features
track_analysis = track_analysis[["request_id", "request_time", "mode", "camelot", "camelot_x", "camelot_y", "is_major", "duration", "tempo", "popularity", "energy", "happiness", "acousticness", "danceability", "instrumentalness", "speechiness", "liveness", "loudness"]]
track_history  = track_history[["request_id", "request_time", "title", "artist", "album", "duration", "program_start_ts", "plays", "is_ad"]]
track_metadata = track_metadata.drop(columns=["id", "artist_sort_name", "release_statuses", "release_languages", "release_scripts", "release_qualities"])

# one-hot encode genres, just in case.
genres = pd.concat([track_metadata[["request_id", "request_time"]], track_metadata["genres"].str.get_dummies(sep=" | ")], axis=1).drop("|", axis=1)

# combine tables
data = pd.merge(track_analysis, track_metadata, on=["request_id", "request_time"], how="inner")
data = pd.merge(data, track_history, on=["request_id", "request_time"], how="inner").drop(0)
# data = pd.merge(data, genres, on=["request_id", "request_time"], how="inner")
data.head()

,request_id,request_time,mode,camelot,camelot_x,camelot_y,is_major,duration_x,tempo,popularity,...,release_dates,release_countries,release_genres,title_y,artist,album,duration_y,program_start_ts,plays,is_ad
1,01a15c73-d8a6-747f-aa75-cb4fb88031f9,2026-10-21 03:14:18.406577+00:00,major,3B,6.123234e-17,1.000000,1,234000,84,46,...,2003-08-18 | 2003-08-26 | 2003 | 2004-05-11 | ...,GB | XE | US | TW | DE | XW,art pop | avant-garde | electronica | experime...,Enjoy,Björk,Post,234,2026-10-21 03:14:13+00:00,1,0
2,01a15cec-dd59-73fc-b30c-31f9cf2a0d89,2026-10-21 05:26:29.465178+00:00,major,3B,6.123234e-17,1.000000,1,194000,108,64,...,2017-06-14 | 2017-12-07 | 2018-06-15 | 2019-07...,AF | AL | XW | GB,dance-pop | electronic | pop | bubblegum pop |...,Ponyboy,Sophie,OIL OF EVERY PEARL'S UN-INSIDES,194,2026-10-21 05:25:59+00:00,4,0
3,01a15d13-2596-77d0-9f2b-c07761a8e47f,2026-10-21 06:08:18.326578+00:00,major,3B,6.123234e-17,1.000000,1,201000,89,64,...,2025-07-11,GB,electronic | garage house,All My Friends,Barry Can't Swim,Loner,200,2026-10-21 06:07:23+00:00,6,0
4,01a15d15-9ced-72e0-9e2f-186d6be8fd86,2026-10-21 06:10:59.949102+00:00,major,7B,-8.660254e-01,-0.500000,1,182000,170,50,...,2025-09-19 | 2026-09-18,XW | US | JP,alternative rock | country rock | grunge | ind...,Reality TV Argument Bleeds [Clean Edit],Wednesday,Bleeds,181,2026-10-21 06:10:44+00:00,17,0
5,01a15d23-a643-71ae-ab21-922bb9533947,2026-10-21 06:26:19.843196+00:00,minor,2A,5.000000e-01,0.866025,0,178000,104,34,...,2007-01-16 | 2007-04-09 | 2007-04-24 | 2007 | ...,US | GB | XE,funk | r&b | soul,Runnin' Away (single version),Sly & The Family Stone,There's a Riot Goin' On,159,2026-10-21 06:25:44+00:00,1,0


**Note.** The camelot wheel is a circle, so standard linear numbers can confuse ML models. Specifically, a model might think 12 and 1 are far apart, even though they are neighbors. The solution used here is projecting the 12 positions onto a 2D coordinate system using cos/sin.

For example, 12 and 1 on the camelot wheel are in key (and should therefore have a small "distance"). 12 lies at point (1, 0); 1 at (.866, 0.5). Using euclidian distance, the distance between them is 0.51; the distance between 12 and 6 (opposite sides of the wheel) is 2.0, which we can use to describe that that 12 and 1 are very closely related, while 12 and 6 are mathematically distant (opposites on the camelot wheel).